### BERTopic
1. 임베딩
2. 차원축소(PCA)
3. 군집(KMeans)
4. 대표어추출

#### c-TF-IDF
- 관점을 바꿨다! 
    -> 한 토픽에 속한 문서 전부를 하나의 큰 문서처럼 합친 후, 그 토픽을 다른 토픽과 구별해주는 단어를 뽑는 방식

- 예)
    - 반도체 토픽에서 삼성, 반도체, 수출 -> 자주 나오고, 다른 토픽엔 잘 안나온다면
    - 이 단어가 그 토픽의 대표어가 되는 방식

#### 고전적 토픽 모델
- 단어 빈도(bag of words)
- 한국어 -> 형태소 분석기가 같이 필요
- 토픽 숫자 정해야 함

#### BERTopic
- 임베딩 기반
- 토픽 수도 자동으로 결정
- 다국어, 한국어 지원
- 짧은 글, 유의어 -> 가능

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "dragonkue/BGE-m3-ko",
    device="cpu"
)

data = pd.read_csv("../data/11-1_뉴스정제.csv")
data.head(3)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...


In [2]:
from bertopic.backend import BaseEmbedder

In [3]:
class E5KoreanEmbedder(BaseEmbedder):
    def __init__(self):
        super().__init__()
        self.model = SentenceTransformer(
            "dragonkue/BGE-m3-ko",
            device="cpu")
    def embed(self, texts, verbose=False):
        return self.model.encode(
            texts,
            batch_size=32, normalize_embeddings=True, show_progress_bar=False,
        )

embedder = E5KoreanEmbedder()
print("임베더 준비 완료")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

임베더 준비 완료


In [4]:
# c-TF-IDF 구하기 -> 키워드를 가져와야 함 -> 명사
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer

kiwi = Kiwi()

# 가져올 명사, 동사, 형용사, 외국어 태그를 지정
keep_pos = {"NNG", "NNP", "VV", "VA", "SL"}
stop_words = {"기자", "뉴스", "사진", "제공", "관련", "대해", "통해", "위해"}

In [ ]:
def kiwi_tokenize(text):
    return [t.form for t in kiwi.tokenize(text)
            if t.tag in keep_pos and len(t.form) > 1 and t.form not in stop_words]

vectorizer = CountVectorizer(
    tokenizer=kiwi_tokenize, token_pattern=None, lowercase=False,
    ngram_range=(1, 2), min_df=2, max_df=0.9,
)

#### 차원 축소, 군집 설정
- UMAP
- HDBSCAN

In [7]:
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                metric="cosine", random_state=42, low_memory=True)
hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=5, metric="euclidean",
                     cluster_selection_method="eom", prediction_data=False)
print("UMAP·HDBSCAN 준비 완료")

UMAP·HDBSCAN 준비 완료


In [10]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

docs = [d.strip() for d in data["정제본문"].tolist() if isinstance(d, str) and d.strip()][:100]

# 안에 임베딩, 토큰, 군집화까지 다 해서 top word 분류하는 상황!
topic_model = BERTopic(
    embedding_model=embedder,       # 임베딩
    vectorizer_model=vectorizer,    # 벡터라이징
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,    # 군집
    representation_model=KeyBERTInspired(),
    calculate_probabilities=False,
    top_n_words=8,
    verbose=False,
)

topics, _ = topic_model.fit_transform(docs)
print("학습 완료. 문서마다 토픽 번호가 매겨졌습니다.")

학습 완료. 문서마다 토픽 번호가 매겨졌습니다.


In [14]:
info = topic_model.get_topic_info()
info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,11,-1_특가_할인_분양 예정_예정 밝히,"[특가, 할인, 분양 예정, 예정 밝히, 보험 가입, 금액, G마켓, 현재, 설문,...",[10일까지 국내외 여행 상품 특가 판매 여행 빅세일 이베이코리아 제공 뉴스1 서울...
1,0,45,0_하반기_공급_상반기_물량,"[하반기, 공급, 상반기, 물량, 인플레이션, 거래, 분석, 규제, 현재, 지구]",[기사내용 요약 가계대출 잔액 699조6521억원 금리상승 자산시장 주춤 영향 하반...
2,1,44,1_할인_할인 혜택_회사_공급,"[할인, 할인 혜택, 회사, 공급, 배송, 박람회, 공장, 중소기업, 감축, 공단]",[에너지업계 최신 이슈 소개 친환경 콘텐츠 집중 유 부회장 넷제로 기업 노력 정부 ...


In [16]:
info[['Topic', 'Count', 'Name']]

,Topic,Count,Name
0,-1,11,-1_특가_할인_분양 예정_예정 밝히
1,0,45,0_하반기_공급_상반기_물량
2,1,44,1_할인_할인 혜택_회사_공급


In [21]:
# 주요 토픽은 -1이 아닌 것
real_topic = [t for t in info['Topic'] if t != -1]
real_topic

[0, 1]